In [ ]:
# TASK 1.1   Read the file the foundations way
##################################################

import csv
import json


# Step 1: Open CSV file and read using DictReader
with open('appointments.csv', 'r',newline="") as file:
    reader = csv.DictReader(file)
    header = next(reader)
    print(f"Header: ",reader.fieldnames)

   #Count total rows
    rows = list(reader)
    print("Row count:", len(rows))


# Step 2: Function to convert data types
def parse_row(row):
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])

    return row

# Step 3: Create accepted and rejected lists
accepted = []
rejected = []

# Step 4: Loop through every row
for row in rows:
    try:
        converted_row = parse_row(row)
        accepted.append(converted_row)

    except ValueError:
        rejected.append(row)


# Step 5: Save rejected rows into JSON file
with open('rejected_rows.json', 'w',newline="") as file:
    json.dump(rejected, file, indent=2)


# Step 6: Print summary
print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))
print("Rejected rows saved to rejected_rows.json")

##############################################################################################




In [ ]:
# TASK 1.2   Inspect and clean, pass 1
########################################



import pandas as pd

df=pd.read_csv("/content/appointments.csv")
print("shape:",df.shape)
print("Data info:",df.info())
print("duplicates:",df.duplicated().sum())
print("Missing Values:",  df.isnull().sum())

# A)  Drop duplicate rows

print("Before drop duplicates,row counts:",len(df))
df.drop_duplicates(inplace=True)
print("After drop duplicates,row counts:",len(df))


# B) Clean booking_channel

# Before cleaning
print("Distinct raw values:", df["booking_channel"].nunique())
#  Clean booking_channel
df['booking_channel'] = df['booking_channel'].str.strip().str.title()
# After cleaning
print("Distinct cleaned values:", df["booking_channel"].nunique())


print(df["booking_channel"].unique())



# C ) Convert appt_date and booked_on  to date_time format
# Using errors='coerce' to handle invalid strings like 'not booked'
df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
df['booked_on'] = pd.to_datetime(df['booked_on'], errors='coerce')

print("After date conversion,Row counts:", len(df))



# D: Drop rows with missing appt_date or status

print("Missing appointment dates:", df["appt_date"].isnull().sum())
print("Missing statuses:", df["status"].isnull().sum())

df = df.dropna(subset=["appt_date", "status"])



print("After dropping missing dates and status,row counts:", len(df))

print( "Final Shape:",df.shape)




In [ ]:
# Task 1.3 — Clean, pass 2: validity, gaps and outliers
###########################################################

import numpy as np
import pandas as pd



# (a) Treat age below 1 or above 100 as invalid

invalid_age_count = ((df["age"] < 1) | (df["age"] > 100)).sum()
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

print("Invalid ages:", invalid_age_count)
print("Total missing ages:", df["age"].isnull().sum())


# (b) Fill missing values with median

age_median = df["age"].median()
distance_median = df["distance_km"].median()
df["age"] = df["age"].fillna(age_median).round().astype(int)

df["distance_km"] = df["distance_km"].fillna(
    distance_median
)
print("Median age:", age_median)
print("Median distance:", distance_median)


# (c) Calculate Q1, Q3 and IQR and cap outliers

# Distance_km
q1_distance = df["distance_km"].quantile(0.25)
q3_distance = df["distance_km"].quantile(0.75)

iqr_distance = q3_distance - q1_distance

lower_distance = q1_distance - (1.5 * iqr_distance)
upper_distance = q3_distance + (1.5 * iqr_distance)


# Count values to be capped
distance_capped = (
    (df["distance_km"] < lower_distance) |
    (df["distance_km"] > upper_distance)
).sum()

low_distance_count = (df["distance_km"] < lower_distance).sum()

df["distance_km"] = np.clip(
    df["distance_km"],
    lower_distance,
    upper_distance
)

print("\n--- Distance Outlier Results ---")
print("Q1:", q1_distance)
print("Q3:", q3_distance)
print("IQR:", iqr_distance)
print("Lower fence:", lower_distance)
print("Upper fence:", upper_distance)
print("Values capped:", distance_capped)
print("Low-side values:", low_distance_count)



# Fee_inr
q1_fee = df["fee_inr"].quantile(0.25)
q3_fee = df["fee_inr"].quantile(0.75)

iqr_fee = q3_fee - q1_fee

lower_fee = q1_fee - (1.5 * iqr_fee)
upper_fee = q3_fee + (1.5 * iqr_fee)

# Count values to be capped
fee_capped = (
    (df["fee_inr"] < lower_fee) |
    (df["fee_inr"] > upper_fee)
).sum()

df["fee_inr"] = np.clip(
    df["fee_inr"],
    lower_fee,
    upper_fee
)

print("\n--- Fee Outlier Results ---")
print("Q1:", q1_fee)
print("Q3:", q3_fee)
print("IQR:", iqr_fee)
print("Lower fence:", lower_fee)
print("Upper fence:", upper_fee)
print("Values capped:", fee_capped)

# (d) Round values
df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round().astype(int)


print("\n--- Final Results ---")
print("Largest fee afterwards:", df["fee_inr"].max())
print("Final row count:", len(df))


In [ ]:
# Task 1.4 — Features and a NumPy summary
##########################################

import numpy as np

# Create lead_days
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days

# Create no_show
df['no_show'] = (df['status'] == 'No-Show').astype(int)

# Numpy array
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# a) Column Mean
column_means = np.mean(X, axis=0)
print("Column Means:", column_means)

# b) lead_days >14

mask = df['lead_days'] > 14

print("\nLead days:")
print("Minimum:", df["lead_days"].min())
print("Maximum:", df["lead_days"].max())
print("Rows with lead_days > 14:", mask.sum())
print("Share:", round(mask.mean() * 100, 1), "%")



# No-show statistics
print("\nNo-show statistics:")
print("Total no-shows:", df["no_show"].sum())
print("Total appointments:", len(df))
print("No-show percentage:", round(df["no_show"].mean() * 100, 1), "%")


# (c) Distance z-scores
x = df["distance_km"].to_numpy()
z=(x-np.mean(x))/np.std(x)
print("\nDistance z-score:")
print("Rows with |z| > 2:", (np.abs(z) > 2).sum())



In [ ]:
# Task 1.5 — Enrich the clinics from a live API
################################################

import pandas as pd
import requests

# Step 1: Load clinics CSV
clinics_df = pd.read_csv("clinics.csv")

# Step 2: Create a list to store temperatures
temperatures = []

# Step 3: Loop through all 8 clinics
for index, row in clinics_df.iterrows():

    temp_c = None
    reason = "Unknown error"

    # Retry up to 2 times
    for attempt in range(2):

        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": row["latitude"],
                    "longitude": row["longitude"],
                    "hourly": "temperature_2m",
                    "forecast_days": 1
                },
                timeout=10
            )

            # Check HTTP status
            if response.status_code != 200:
                reason = f"HTTP status {response.status_code}"
                continue

            # Read JSON response
            data = response.json()

            # Check hourly temperature data
            hourly = data.get("hourly", {})
            temp_list = hourly.get("temperature_2m")

            if not temp_list:
                reason = "Hourly temperature data missing or empty"
                continue

            # Take the first temperature (midnight UTC)
            temp_c = temp_list[0]

            if temp_c is None:
                reason = "First temperature value is missing"
                continue

            # Successful response
            break

        except requests.exceptions.RequestException as e:
            reason = str(e)

        except (ValueError, TypeError) as e:
            reason = str(e)

    # If both attempts fail
    if temp_c is None:
        print("Failed:", row["clinic_name"], "-", reason)

    temperatures.append(temp_c)


# Step 4: Add temperature column
clinics_df["temp_c"] = temperatures


# Step 5: Save enriched CSV
clinics_df.to_csv("clinics_enriched.csv", index=False)


# Step 6: Display results
# Verify enriched CSV
result = pd.read_csv("clinics_enriched.csv")

print("Total rows:", len(result))
print("Total columns:", len(result.columns))

print("\nTemperature results:")
print(result[["clinic_name", "temp_c"]])

print("\nSuccessful API responses:", result["temp_c"].notna().sum())
print("Missing temperatures:", result["temp_c"].isna().sum())

# Check temperature range
valid_temps = result["temp_c"].dropna()

if len(valid_temps) > 0:
    print("Minimum temperature:", valid_temps.min())
    print("Maximum temperature:", valid_temps.max())

# Final validation
print("\nValidation:")
print("8 clinics:", len(result) == 8)
print("Temperature range valid:",
      valid_temps.between(-10, 50).all())

In [ ]:
#Task 1.6 — Export the clean table and write the SQL


import pandas as pd
import numpy as np

# --------------------------------------------------
# STEP 1: Define the exact 16 appointment columns
# --------------------------------------------------

appointment_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]


# --------------------------------------------------
# STEP 2: Select and sort appointment data
# --------------------------------------------------

appointments = df[appointment_columns].copy()

appointments = appointments.sort_values(
    by="appointment_id"
).reset_index(drop=True)


# --------------------------------------------------
# STEP 3: Check appointment_id uniqueness
# --------------------------------------------------

is_unique = appointments["appointment_id"].is_unique

print("Appointment ID is unique:", is_unique)


# --------------------------------------------------
# STEP 4: Check missing values in all 16 columns
# --------------------------------------------------

missing_values = appointments.isnull().sum()

#print("\nMissing values in each column:")
#print(missing_values)

print("\nTotal missing values:", missing_values.sum())


# Stop if validation fails
if not is_unique or missing_values.sum() > 0:
    raise ValueError("Appointment data failed validation!")


# --------------------------------------------------
# STEP 5: Format date columns as YYYY-MM-DD
# --------------------------------------------------

appointments["booked_on"] = pd.to_datetime(
    appointments["booked_on"]
).dt.strftime("%Y-%m-%d")

appointments["appt_date"] = pd.to_datetime(
    appointments["appt_date"]
).dt.strftime("%Y-%m-%d")


# --------------------------------------------------
# STEP 6: Save appointments_clean.csv
# --------------------------------------------------

appointments.to_csv(
    "appointments_clean.csv",
    index=False
)

print("\nappointments_clean.csv saved successfully")
print("Appointment rows:", len(appointments))
print("Appointment columns:", len(appointments.columns))


# --------------------------------------------------
# STEP 7: Load enriched clinics data
# --------------------------------------------------

clinics = pd.read_csv("clinics_enriched.csv")

clinic_columns = [
    "clinic_id",
    "clinic_name",
    "city",
    "tier",
    "latitude",
    "longitude",
    "temp_c"
]

clinics = clinics[clinic_columns]


# --------------------------------------------------
# STEP 8: Function to convert Python values to SQL
# --------------------------------------------------

def sql_value(value):

    if pd.isna(value):
        return "NULL"

    if isinstance(value, (bool, np.bool_)):
        return "1" if value else "0"

    if isinstance(value, (int, np.integer)):
        return str(int(value))

    if isinstance(value, (float, np.floating)):
        return str(float(value))

    return f"'{value}'"


# --------------------------------------------------
# STEP 9: Write SQL INSERT statements
# Clinics first, appointments second
# --------------------------------------------------

with open("load_data.sql", "w") as file:

    # Insert clinic records first
    for _, row in clinics.iterrows():

        values = [
            sql_value(row[column])
            for column in clinic_columns
        ]

        sql = f"INSERT INTO clinics VALUES ({','.join(values)});\n"

        file.write(sql)


    # Insert appointment records next
    for _, row in appointments.iterrows():

        values = [
            sql_value(row[column])
            for column in appointment_columns
        ]

        sql = f"INSERT INTO appointments VALUES ({','.join(values)});\n"

        file.write(sql)


# --------------------------------------------------
# STEP 10: Final summary
# --------------------------------------------------

print("\n--- EXPORT SUMMARY ---")

print("Clinics exported:", len(clinics))
print("Appointments exported:", len(appointments))

print("Total SQL INSERT statements:",
      len(clinics) + len(appointments))

print("Files created:")
print("1. appointments_clean.csv")
print("2. load_data.sql")


In [ ]:
from google.colab import files

#files.download('load_data.sql')

In [ ]:
# Part 2 — SQL: verify the numbers and find where no-shows cluster
#Task 2.2 — No-show rate by clinic


import pandas as pd

# Load clinic data
clinics_df = pd.read_csv("clinics_enriched.csv")

# Group appointments by clinic_id
clinic_summary = df.groupby("clinic_id").agg(
    appointments=("appointment_id", "count"),
    no_shows=("no_show", "sum")
).reset_index()

# Join clinic names and cities
clinic_summary = clinic_summary.merge(
    clinics_df[["clinic_id", "clinic_name", "city"]],
    on="clinic_id",
    how="left"
)

# Calculate no-show percentage
clinic_summary["no_show_pct"] = (
    100.0 * clinic_summary["no_shows"] /
    clinic_summary["appointments"]
).round(1)

# Sort from highest to lowest
clinic_summary = clinic_summary[
    ["clinic_name", "city", "appointments", "no_shows", "no_show_pct"]
].sort_values(by="no_show_pct", ascending=False)

print(clinic_summary.to_string(index=False))

print("Total appointments:", clinic_summary["appointments"].sum())
print("Total no-shows:", clinic_summary["no_shows"].sum())
print("Number of clinics:", len(clinic_summary))